# 01 · Principal-risk extraction

**Step 1 of the pipeline.** Read report pages as images and return every principal risk on them, once per page it
appears on. Merging the same risk across pages is step 2 (notebook 02); evaluating this step against the golden set
is its own notebook.

- **Why images, not the text layer:** the text layer loses reading order and the layout cues (tables, legends,
  icons) that decide whether an item is a risk, an opportunity or an impact. See `00_parsing_comparison.ipynb`.
- **Why several pages per call:** a table or a risk description often continues onto the next page. Adjacent pages
  are sent together and the model says which page each risk is on.
- **Why copy, not summarise:** description and mitigation are copied word for word, so every risk stays grounded in
  the page (plan: consistency and grounding). Summarising is step 2's job.

The code lives in `annual_report_reader/extraction/`, one module per section (named in its heading). This
notebook shows each stage working on the real report; the modules hold the definitions and are unit-tested
(`tests/extraction/`).

## 0. The principal-risk definition

What counts as a principal risk is a spec shared by the prompt, the golden set and the judge:
`prompts/principal-risk-definition.md`. Edit it there.

In [ ]:
from IPython.display import Markdown, display

from annual_report_reader.extraction.config import REPO_ROOT

display(Markdown((REPO_ROOT / "prompts/principal-risk-definition.md").read_text(encoding="utf-8")))

## 1. Config → `extraction/config.py`

Everything that decides what a run produces lives in one frozen object, so a run can be saved with its exact
settings and repeated. Pages are sorted and must be unique, since grouping (section 3) relies on their order. Paths are relative to the repo root, so a saved run does not depend on the machine.
Secrets (endpoint, API key) stay in `.env` and never enter the config. The deployment name is not a secret and is
part of the config, because the model decides the output.

In [ ]:
from dotenv import load_dotenv

from annual_report_reader.extraction.config import ExtractionConfig

load_dotenv(REPO_ROOT / ".env")  # endpoint, key and deployment name

CONFIG = ExtractionConfig()
CONFIG

## 2. Schema → `extraction/schema.py`

What one call returns. The fields match the golden set (`eval/golden/risk_extraction.yaml`), so the output can be
evaluated without conversion. The schema is sent as strict structured output, so every field is required and
"no value" is an explicit `null` (or `"None"` for mitigation, as the prompt asks).

Categories are the case's examples plus `other`. `confidence` lets the analyst sort borderline items: the prompt
asks for `low` when an item is close to an impact, an opportunity or a generic statement (plan: recall first,
precision is paid by the analyst).

In [ ]:
# The JSON schema the model is constrained to.
import json

from annual_report_reader.extraction.schema import ExtractedRisks

print(json.dumps(ExtractedRisks.model_json_schema(), indent=1)[:1500], "...")

## 3. Pages → `extraction/pages.py`

Pages are rendered to PNG **before** any call, sequentially: PyMuPDF is not thread-safe, and this way the calls only
ever receive ready, immutable data. Rendering a handful of pages takes about a second.

Each run of adjacent pages is sent as **one call** (e.g. 50-51, 71-74), with no size cap. A table or risk
description that continues onto the next page is then read as one, and the model says which page each risk is on.
A run is short in practice: a principal-risk section rarely spans more than a few pages.

In [ ]:
# Render the configured pages and show the groups.
from IPython.display import Image

from annual_report_reader.extraction.pages import adjacent_runs, render_pages

images = render_pages(CONFIG.resolve(CONFIG.pdf_path), CONFIG.pages, CONFIG.dpi)
groups = adjacent_runs(images)
for g in groups:
    print(f"{g.label:<8} {len(g.images)} page(s), {sum(len(i.png) for i in g.images) / 1e6:.1f} MB")

display(Image(images[1].png, width=600))  # p.51: the main-risks page

## 4. Prompt → `extraction/prompts.py`

The system prompt is a file, `prompts/<prompt>.md`; every saved run also stores the exact prompt sent.
`{PRINCIPAL_RISK_DEFINITION}` is filled in from the spec, so the definition and the
prompt change independently.

The prompt is a `ChatPromptTemplate`: the system message, then a placeholder for the page message. The number of
pages varies per call, which a fixed template cannot express, so `pages_message` builds that message: each image
is preceded by "Page N", which is how the model knows the page number of each risk.

In [ ]:
# The messages sent for the first group (image data cut short).
from annual_report_reader.extraction.prompts import load_prompt, pages_message

prompt = load_prompt(
    CONFIG.resolve(CONFIG.prompt_dir),
    CONFIG.prompt,
    CONFIG.resolve(CONFIG.definition_path).read_text(encoding="utf-8"),
)
for m in prompt.invoke({"pages": [pages_message(groups[0])]}).to_messages():
    if isinstance(m.content, str):
        print(f"--- {m.type} ({len(m.content):,} chars)\n{m.content[:600]}\n[...]")
    else:
        parts = [
            p["text"] if p["type"] == "text" else f"<image {len(p['image_url']['url']) / 1e6:.1f} MB>"
            for p in m.content
        ]
        print(f"--- {m.type}\n" + " | ".join(parts))

## 5. Chain → `extraction/chain.py`, `chat.py`

`AzureChatOpenAI` with **strict JSON-schema structured output**: the response is constrained to `ExtractedRisks`,
so it is always valid JSON in the right shape. `include_raw=True` also returns the raw message, which carries the
token usage (including reasoning tokens) and any parsing error.

The chain takes a `PageGroup` and builds the page message itself, so the same chain runs one group (`invoke`) or
all of them (`batch`). Endpoint and key are read from `.env` by `AzureChatOpenAI` (`AZURE_OPENAI_ENDPOINT`,
`AZURE_OPENAI_API_KEY`). Transient errors (rate limits, 5xx) are retried by the client with backoff.

Each call becomes a `CallResult`: the pages sent, the risks, the usage, or the error. A failed call is a result too,
so one bad call never loses the others.

In [ ]:
# One call on the smallest group (p.85).
from annual_report_reader.chat import azure_chat_model
from annual_report_reader.extraction.chain import call_result, extraction_chain

chain = extraction_chain(prompt, azure_chat_model(CONFIG.deployment, CONFIG.reasoning_effort))
group = next(g for g in groups if g.pages == [85])
one = call_result(group, chain.invoke(group))
print(f"{group.label}: {len(one.risks)} risks | {one.usage} | ${CONFIG.cost(one.usage):.4f} | error: {one.error}")
for r in one.risks:
    print(f"  p.{r.page} [{r.category}/{r.secondary_category}, {r.confidence}] {r.title}")

## 6. Run all groups → `extraction/chain.py`

`batch` runs the groups concurrently (`max_concurrency`) and, with `return_exceptions=True`, returns an exception in
place of a failed group instead of stopping the run.

In [ ]:
import time
from datetime import UTC, datetime

from annual_report_reader.chat import Usage, tracing_callbacks
from annual_report_reader.extraction.chain import extract_groups
from annual_report_reader.extraction.runs import run_id_for

started_at = datetime.now(UTC)
run_id = run_id_for(started_at)
start = time.perf_counter()
results = extract_groups(chain, groups, CONFIG, run_id)
elapsed = time.perf_counter() - start

for g, res in zip(groups, results):
    status = f"FAILED: {res.error}" if res.error else f"{len(res.risks):2} risks"
    print(
        f"{g.label:<8} | {status} | in {res.usage.input_tokens:,} / out {res.usage.output_tokens:,} "
        f"(reasoning {res.usage.reasoning_tokens:,}) | ${CONFIG.cost(res.usage):.4f}"
    )
total = sum((r.usage for r in results), Usage())
print(
    f"{run_id}: {sum(len(r.risks) for r in results)} risks | {len(results)} calls | {elapsed:.0f}s | "
    f"${CONFIG.cost(total):.4f} | tracing: {'on' if tracing_callbacks() else 'off'}"
)

## 7. Save the run → `extraction/runs.py`

A run is saved as one JSON file in `runs/extract/`: the config, the exact system prompt sent, and every call with
its risks, usage or error. It is the input of the next steps (merge, evaluation) and a record that can be compared
later: every attempt is kept, including failed ones.

The file is written and read by the `ExtractionRun` model, so it always loads back into the same types.

In [ ]:
from annual_report_reader.extraction.runs import RUNS_DIR, ExtractionRun, save_run
from annual_report_reader.extraction.schema import ExtractedRisks

run = ExtractionRun(
    run_id=run_id,
    started_at=started_at,
    elapsed_s=round(elapsed, 1),
    config=CONFIG,
    system_prompt=prompt.format_messages(pages=[])[0].content,
    output_schema=ExtractedRisks.model_json_schema(),
    calls=results,
)
run_path = save_run(run, REPO_ROOT / RUNS_DIR)
print(f"Saved {run_path.relative_to(REPO_ROOT)}")

## 8. Inspect → `extraction/runs.py`

Read back from the file just saved. Every configured page is listed, including pages with no risks: p.50 and p.72
are in the golden set **because** they have none (false-positive tests). A page the model named but that was not
sent would show up as `not sent`.

In [ ]:
from annual_report_reader.extraction.runs import load_run, risks_per_page

loaded = load_run(run_path)
print(
    f"{loaded.run_id}: {len(loaded.risks)} risks | {len(loaded.calls)} calls | {loaded.elapsed_s:.0f}s | "
    f"${loaded.config.cost(loaded.usage):.4f}"
)
risks_per_page(loaded).T

## 9. Observations

_To fill in after reviewing the run._